In [1]:
# Check the path of Python interpreter
!which python

/d/Codding/Projects/ICDS/.venv/Scripts/python


In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Alignment, PatternFill, Font, Border, Side
from openpyxl.utils import get_column_letter

def generate_day_names(month, year):
    # Generate all dates in the given month and year
    date_range = pd.date_range(start=f'{year}-{month:02d}-01', end=f'{year}-{month:02d}-{pd.Timestamp(year, month, 1).days_in_month}')
    
    # Create DataFrame with day numbers, day names, and additional columns
    df = pd.DataFrame({
        ('', 'Date'): date_range.day,
        ('', 'Days'): date_range.strftime('%A'),
        ('', 'Pregnant (Mothers)'): '',
        ('', 'Lactating (Mothers)'): '',
        ('', 'Boys (6m - 3y)'): '',
        ('', 'Girls (6m - 3y)'): '',
        ('', 'Boys (3y - 6y)'): '',
        ('', 'Girls (3y - 6y)'): '',
    })
    
    return df

# Generate DataFrame for the given month and year
month = 3
year = 2025
polio_day = 16  # Day of the month for Polio campaign

# List of holidays
holidays = {
    '26 January 2025': 'Republic Day',
    '07 March 2025': 'Doljatra',
    '15 April 2025': 'Bengali New Year’s Day',
    '22 April 2025': 'Id-Ul-Fitre',
    '29 June 2025': 'Id-Ud-Zoha',
    '29 July 2025': 'Muharram',
    '15 August 2025': 'Independence Day',
    '21 October 2025': 'Durga Puja (Saptami)',
    '23 October 2025': 'Durga Puja (Nabami)',
    '24 October 2025': 'Durga Puja (Dashami)',
    '15 November 2025': 'Bhatirdwitiya',
    '25 December 2025': 'Christmas Day'
}

In [ ]:
# month = int(input('Enter month number: '))
# year = int(input('Enter year: '))
# polio_campaign = input('Is there a Polio Campaign scheduled for this month? (Yes/No): ').strip().lower()
# if polio_campaign in ['y', 'yes']:
#     polio_day = int(input('Enter Polio campaign day: '))
#     print(f"Polio campaign scheduled on {year}-{month:02d}-{polio_day:02d}")
# else:
#     polio_day = None
#     print("No polio campaign scheduled.")

No polio campaign scheduled.


In [4]:

df = generate_day_names(month, year)

# Flatten the MultiIndex columns for easier handling in Excel
df.columns = [' '.join(col).strip() for col in df.columns.values]

# Generate dynamic filename based on month and year
month_name = pd.Timestamp(year, month, 1).strftime('%B')
output_filename = f"{month_name}_{year}_Report_Format.xlsx"
output_path = fr"C:\Users\MUHAMMAD NOUKHEZ\Downloads\{output_filename}"

# Initialize a new workbook and add a worksheet
wb = Workbook()
ws = wb.active
ws.title = "Row_sheet"

# Set header row with column names
for row in dataframe_to_rows(df, index=False, header=True):
    ws.append(row)

# Define styles
header_fill = PatternFill(start_color="ADD8E6", end_color="ADD8E6", fill_type="solid")  # Light blue header background
header_font = Font(bold=True)  # Bold font for headers
light_red_fill = PatternFill(start_color="FFC0C0", end_color="FFC0C0", fill_type="solid")  # Light red fill for Sundays
light_green_fill = PatternFill(start_color="C0FFC0", end_color="C0FFC0", fill_type="solid")  # Light green fill for holidays
center_align = Alignment(horizontal="center", vertical="center")
left_align = Alignment(horizontal="left", vertical="center")
header_rotation = Alignment(horizontal="center", vertical="bottom", textRotation=90)  # Rotate header and bottom align

# Define border style
thin_border = Border(
    left=Side(style="thin"),
    right=Side(style="thin"),
    top=Side(style="thin"),
    bottom=Side(style="thin")
)

# Apply header styles with rotation, bottom alignment, and borders
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = header_rotation
    cell.border = thin_border

# Apply styles and borders for each row
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
    date_cell = row[0]  # Date's column (first column in each row)
    day_cell = row[1]  # Day's column (second column in each row)
    
    # Format date as "DD Month YYYY"
    full_date = f"{date_cell.value:02d} {month_name} {year}"
    
    # Check if it's a holiday
    if full_date in holidays:
        holiday_name = holidays[full_date]
        for cell in row:
            cell.fill = light_green_fill
            cell.alignment = center_align
            cell.border = thin_border
        
        # Merge cells after "Days" column to the last column and insert holiday name
        start_col = get_column_letter(3)  # First column after "Days"
        end_col = get_column_letter(ws.max_column)
        ws.merge_cells(f"{start_col}{date_cell.row}:{end_col}{date_cell.row}")
        
        # Insert holiday name in the first merged cell
        ws[f"{start_col}{date_cell.row}"].value = holiday_name
        ws[f"{start_col}{date_cell.row}"].alignment = center_align
        ws[f"{start_col}{date_cell.row}"].font = Font(bold=True)

    elif day_cell.value == "Sunday" and date_cell.value != polio_day:
        # Apply light red fill, center alignment, and border to each cell in the row for Sundays
        for cell in row:
            cell.fill = light_red_fill
            cell.alignment = center_align
            cell.border = thin_border
        
        # Merge cells after "Days" column for Sundays
        start_col = get_column_letter(3)  # First column after "Days"
        end_col = get_column_letter(ws.max_column)
        ws.merge_cells(f"{start_col}{day_cell.row}:{end_col}{day_cell.row}")
    
    else:
        if date_cell.value == polio_day:
            for cell in row:
                cell.fill = light_red_fill
        # Apply center alignment and border to all cells except the "Day's" column, which is left-aligned
        for cell in row:
            cell.alignment = center_align
            cell.border = thin_border
        day_cell.alignment = left_align  # Left-align the "Day's" column

# Set specific column widths
ws.column_dimensions[get_column_letter(1)].width = 4.5  # Date column
ws.column_dimensions[get_column_letter(2)].width = 10.5 # Day's column
for col_idx in range(3, ws.max_column + 1):
    ws.column_dimensions[get_column_letter(col_idx)].width = 6.5  # Other columns

# Save the workbook
wb.save(output_path)

print(f"File saved as: {output_path}")


File saved as: C:\Users\MUHAMMAD NOUKHEZ\Downloads\March_2025_Report_Format.xlsx
